<h1>Validación 04. Reconstrucción espacial del perfil 3D</h1>

<h2>Objetivo</h2>

<p>
El objetivo de esta validación es comprobar que la transformación de los puntos
del perfil topográfico 2D hacia coordenadas espaciales no introduce un
desplazamiento horizontal.
</p>

<p>
En SecGeol, cada punto del perfil 2D se representa mediante:
</p>

<ul>
    <li>
        la <strong>distancia acumulada</strong> sobre la línea de sección;
    </li>
    <li>
        la <strong>elevación</strong> obtenida del modelo digital de elevación.
    </li>
</ul>

<p>
Para reconstruir el perfil en el espacio tridimensional, la distancia acumulada
se utiliza como argumento de:
</p>

<p style="text-align: center;">
    <code>geom_linea.interpolate(distancia)</code>
</p>

<p>
Esta operación devuelve la posición XY correspondiente sobre la línea guía.
Posteriormente, la elevación del perfil se incorpora como coordenada Z.
</p>

<h2>Comparación propuesta</h2>

<p>
Para cada punto generado por SecGeol se compararán:
</p>

<table style="border-collapse: collapse; margin-top: 10px;">
    <thead>
        <tr>
            <th style="border: 1px solid #999; padding: 6px;">
                Información original
            </th>
            <th style="border: 1px solid #999; padding: 6px;">
                Información reconstruida
            </th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border: 1px solid #999; padding: 6px;">
                Distancia acumulada del perfil 2D
            </td>
            <td style="border: 1px solid #999; padding: 6px;">
                Distancia utilizada por <code>interpolate()</code>
            </td>
        </tr>
        <tr>
            <td style="border: 1px solid #999; padding: 6px;">
                Coordenada X original del vértice densificado
            </td>
            <td style="border: 1px solid #999; padding: 6px;">
                Coordenada X reconstruida
            </td>
        </tr>
        <tr>
            <td style="border: 1px solid #999; padding: 6px;">
                Coordenada Y original del vértice densificado
            </td>
            <td style="border: 1px solid #999; padding: 6px;">
                Coordenada Y reconstruida
            </td>
        </tr>
        <tr>
            <td style="border: 1px solid #999; padding: 6px;">
                Elevación muestreada del DEM
            </td>
            <td style="border: 1px solid #999; padding: 6px;">
                Coordenada Z del perfil 3D
            </td>
        </tr>
    </tbody>
</table>

<h2>Indicadores de validación</h2>

<p>
Se calcularán los siguientes errores para cada punto:
</p>

<ul>
    <li>
        <strong>&Delta;X:</strong> diferencia entre la coordenada X reconstruida
        y la coordenada X original;
    </li>
    <li>
        <strong>&Delta;Y:</strong> diferencia entre la coordenada Y reconstruida
        y la coordenada Y original;
    </li>
    <li>
        <strong>Error XY:</strong> distancia planimétrica entre ambas posiciones;
    </li>
    <li>
        <strong>&Delta;Z:</strong> diferencia entre la elevación original del perfil
        2D y la coordenada Z asignada al punto 3D.
    </li>
</ul>

<div style="
border-left:6px solid #2E75B6;
background:#F4F8FC;
padding:14px;
margin-top:18px;
">

<h3 style="margin-top:0;">Resultado esperado</h3>

<p>
Si los errores XY y Z son nulos o únicamente corresponden a la precisión numérica
del sistema, podrá concluirse que la reconstrucción del perfil 3D conserva
correctamente la posición y la elevación de los puntos del perfil 2D.
</p>

<p>
En ese escenario, el posible desfase observado en la sección final no se origina
durante la reconstrucción espacial del perfil, por lo que la investigación deberá
continuar sobre las etapas de generación de segmentos geológicos, estructuras y
construcción de la geometría final de la sección.
</p>

</div>

### <span style="color:#cc416d">1.Importaciones</span>

In [2]:
from qgis.core import (
    QgsRasterLayer,
    QgsVectorLayer,
    QgsGeometry,
    QgsFeature,
    QgsPoint,
    QgsPointXY,
    QgsWkbTypes
)

import math
import numpy as np

### <span style="color:#cc416d">2.Rutas</span>

In [3]:
ruta_dem = r"C:\Proyectos\2026\seccion\dem2.tif"

ruta_linea_guia = (
    r"C:\Proyectos\2026\seccion\salidas"
    r"\ejemplo_seccion1_guia.shp"
)

ruta_perfil_geologico = (
    r"C:\Proyectos\2026\seccion\salidas"
    r"\poligonos_geol.shp"
)

### <span style="color:#cc416d">3.Cargar capas</span>

In [4]:
dem_layer = QgsRasterLayer(
    ruta_dem,
    "DEM"
)

linea_layer = QgsVectorLayer(
    ruta_linea_guia,
    "Linea_guia",
    "ogr"
)

perfil_geologico_layer = QgsVectorLayer(
    ruta_perfil_geologico,
    "Perfil_geologico_2D",
    "ogr"
)

print("DEM válido:", dem_layer.isValid())
print("Línea guía válida:", linea_layer.isValid())
print(
    "Perfil geológico válido:",
    perfil_geologico_layer.isValid()
)

print("\nCRS DEM:", dem_layer.crs().authid())
print("CRS línea guía:", linea_layer.crs().authid())
print(
    "CRS perfil geológico:",
    perfil_geologico_layer.crs().authid()
)

DEM válido: True
Línea guía válida: True
Perfil geológico válido: True

CRS DEM: EPSG:6368
CRS línea guía: EPSG:6368
CRS perfil geológico: EPSG:6368


### <span style="color:#cc416d">4. Revisar las capas</span>

In [5]:
features_linea = list(
    linea_layer.getFeatures()
)

features_geologia = list(
    perfil_geologico_layer.getFeatures()
)

print(
    "Entidades en línea guía:",
    len(features_linea)
)

print(
    "Polígonos geológicos:",
    len(features_geologia)
)

if not features_linea:
    raise Exception(
        "La capa de línea guía no contiene entidades."
    )

if not features_geologia:
    raise Exception(
        "La capa de perfil geológico no contiene polígonos."
    )

Entidades en línea guía: 1
Polígonos geológicos: 5


### <span style="color:#cc416d">5. Obtener la línea guía</span>

In [6]:
feature_linea = features_linea[0]
geom_linea = feature_linea.geometry()

if geom_linea is None or geom_linea.isEmpty():
    raise Exception(
        "La geometría de la línea guía está vacía."
    )

print("Longitud línea guía:", geom_linea.length())
print("Multipart:", geom_linea.isMultipart())
print(
    "Vértices:",
    len(list(geom_linea.vertices()))
)

Longitud línea guía: 5772.659572948623
Multipart: True
Vértices: 2


### <span style="color:#cc416d">6. Inspeccionar el perfil geológico 2D</span>

In [7]:
print(
    "Tipo geométrico:",
    QgsWkbTypes.displayString(
        perfil_geologico_layer.wkbType()
    )
)

print("\nCampos:")

for campo in perfil_geologico_layer.fields():
    print(
        campo.name(),
        campo.typeName()
    )

extent_perfil = perfil_geologico_layer.extent()

print("\nExtent perfil 2D:")
print("X mínima:", extent_perfil.xMinimum())
print("X máxima:", extent_perfil.xMaximum())
print("Y mínima:", extent_perfil.yMinimum())
print("Y máxima:", extent_perfil.yMaximum())

print(
    "\nLongitud de la línea guía:",
    geom_linea.length()
)

print(
    "Diferencia X máxima - longitud:",
    extent_perfil.xMaximum()
    - geom_linea.length()
)

Tipo geométrico: MultiPolygon

Campos:
id_lito Integer64
tipo String
valor_geo String

Extent perfil 2D:
X mínima: 0.0
X máxima: 5772.659572948555
Y mínima: 86.22000122070312
Y máxima: 699.5700073242188

Longitud de la línea guía: 5772.659572948623
Diferencia X máxima - longitud: -6.730260793119669e-11


### <span style="color:#cc416d">7. Inspeccionar algunos vértices</span>

In [9]:
for i, feat in enumerate(features_geologia[:3]):

    geom = feat.geometry()

    print(f"\nPolígono {i + 1}")
    print("Atributos:", feat.attributes())
    print(
        "Multipart:",
        geom.isMultipart()
    )

    vertices = list(geom.vertices())

    print(
        "Cantidad de vértices:",
        len(vertices)
    )

    print("Primeros vértices:")

    for j, pt in enumerate(vertices[:10]):
        print(
            j,
            f"dist={pt.x():.6f}",
            f"elev={pt.y():.6f}"
        )


Polígono 1
Atributos: [1, 'poligono', '5']
Multipart: True
Cantidad de vértices: 84
Primeros vértices:
0 dist=90.257145 elev=86.220001
1 dist=0.000000 elev=86.220001
2 dist=0.000000 elev=186.559998
3 dist=4.997974 elev=186.410004
4 dist=9.995947 elev=186.410004
5 dist=14.993921 elev=186.850006
6 dist=19.991895 elev=186.800003
7 dist=24.989868 elev=186.220001
8 dist=29.987842 elev=186.490005
9 dist=34.985816 elev=187.000000

Polígono 2
Atributos: [1, 'poligono', '5']
Multipart: True
Cantidad de vértices: 153
Primeros vértices:
0 dist=1223.892105 elev=86.220001
1 dist=90.257145 elev=86.220001
2 dist=395.576213 elev=197.347054
3 dist=399.837892 elev=195.419998
4 dist=404.835866 elev=194.300003
5 dist=409.833840 elev=192.979996
6 dist=414.831813 elev=192.619995
7 dist=419.829787 elev=192.020004
8 dist=424.827761 elev=192.119995
9 dist=429.825734 elev=194.179993

Polígono 3
Atributos: [2, 'poligono', '5']
Multipart: True
Cantidad de vértices: 429
Primeros vértices:
0 dist=3960.523506 elev=

### <span style="color:#cc416d">8. Normalizar la línea guía</span>

In [10]:
from qgis.core import QgsGeometry

if geom_linea.isMultipart():
    partes = geom_linea.asMultiPolyline()

    print("Cantidad de partes:", len(partes))

    if len(partes) != 1:
        raise Exception(
            "La línea guía contiene más de una parte. "
            "Esta validación requiere revisar cómo se ordenan."
        )

    geom_guia_simple = QgsGeometry.fromPolylineXY(
        partes[0]
    )
else:
    geom_guia_simple = QgsGeometry(
        geom_linea
    )

print("Multipart original:", geom_linea.isMultipart())
print("Multipart normalizada:", geom_guia_simple.isMultipart())

print("Longitud original:", geom_linea.length())
print("Longitud normalizada:", geom_guia_simple.length())

print(
    "Diferencia de longitud:",
    geom_guia_simple.length() - geom_linea.length()
)

Cantidad de partes: 1
Multipart original: True
Multipart normalizada: False
Longitud original: 5772.659572948623
Longitud normalizada: 5772.659572948623
Diferencia de longitud: 0.0


### <span style="color:#cc416d">9. Reconstruir todos los vértices</span>

In [11]:
from qgis.core import QgsPointXY

longitud_guia = geom_guia_simple.length()

vertices_reconstruidos = []

contador_global = 0

for indice_poligono, feat in enumerate(
    features_geologia,
    start=1
):
    geom_perfil = feat.geometry()

    for indice_vertice, pt in enumerate(
        geom_perfil.vertices()
    ):
        distancia_perfil = pt.x()
        elevacion_perfil = pt.y()

        tolerancia = 1e-8

        if distancia_perfil < -tolerancia:
            raise Exception(
                f"Distancia negativa en polígono "
                f"{indice_poligono}, vértice {indice_vertice}: "
                f"{distancia_perfil}"
            )

        if distancia_perfil > longitud_guia + tolerancia:
            raise Exception(
                f"Distancia fuera de la línea en polígono "
                f"{indice_poligono}, vértice {indice_vertice}: "
                f"{distancia_perfil}"
            )

        # Corregir únicamente residuos numéricos en los extremos
        distancia_interpolacion = min(
            max(distancia_perfil, 0.0),
            longitud_guia
        )

        punto_interpolado_geom = (
            geom_guia_simple.interpolate(
                distancia_interpolacion
            )
        )

        if punto_interpolado_geom.isEmpty():
            raise Exception(
                f"No fue posible reconstruir el vértice "
                f"{indice_vertice} del polígono "
                f"{indice_poligono}."
            )

        punto_xy = punto_interpolado_geom.asPoint()

        vertices_reconstruidos.append({
            "indice_global": contador_global,
            "poligono": indice_poligono,
            "vertice": indice_vertice,
            "id_lito": feat["id_lito"],
            "valor_geo": feat["valor_geo"],
            "distancia_2d": distancia_perfil,
            "elevacion_2d": elevacion_perfil,
            "x_utm": punto_xy.x(),
            "y_utm": punto_xy.y(),
            "z_3d": elevacion_perfil
        })

        contador_global += 1

print(
    "Vértices 3D reconstruidos:",
    len(vertices_reconstruidos)
)

Vértices 3D reconstruidos: 1182


### <span style="color:#cc416d">10. Revisar el principio y el final</span>

In [12]:
print("Primeros vértices reconstruidos:")

for registro in vertices_reconstruidos[:10]:
    print(
        f'pol={registro["poligono"]}',
        f'vert={registro["vertice"]}',
        f'dist={registro["distancia_2d"]:.6f}',
        f'XY=({registro["x_utm"]:.3f}, '
        f'{registro["y_utm"]:.3f})',
        f'Z={registro["z_3d"]:.3f}'
    )

print("\nÚltimos vértices reconstruidos:")

for registro in vertices_reconstruidos[-10:]:
    print(
        f'pol={registro["poligono"]}',
        f'vert={registro["vertice"]}',
        f'dist={registro["distancia_2d"]:.6f}',
        f'XY=({registro["x_utm"]:.3f}, '
        f'{registro["y_utm"]:.3f})',
        f'Z={registro["z_3d"]:.3f}'
    )

Primeros vértices reconstruidos:
pol=1 vert=0 dist=90.257145 XY=(617237.799, 2113744.082) Z=86.220
pol=1 vert=1 dist=0.000000 XY=(617150.224, 2113765.924) Z=86.220
pol=1 vert=2 dist=0.000000 XY=(617150.224, 2113765.924) Z=186.560
pol=1 vert=3 dist=4.997974 XY=(617155.074, 2113764.715) Z=186.410
pol=1 vert=4 dist=9.995947 XY=(617159.923, 2113763.505) Z=186.410
pol=1 vert=5 dist=14.993921 XY=(617164.772, 2113762.296) Z=186.850
pol=1 vert=6 dist=19.991895 XY=(617169.622, 2113761.086) Z=186.800
pol=1 vert=7 dist=24.989868 XY=(617174.471, 2113759.877) Z=186.220
pol=1 vert=8 dist=29.987842 XY=(617179.321, 2113758.667) Z=186.490
pol=1 vert=9 dist=34.985816 XY=(617184.170, 2113757.458) Z=187.000

Últimos vértices reconstruidos:
pol=5 vert=42 dist=5722.679836 XY=(622702.808, 2112381.051) Z=697.620
pol=5 vert=43 dist=5727.677810 XY=(622707.658, 2112379.841) Z=696.330
pol=5 vert=44 dist=5732.675784 XY=(622712.507, 2112378.632) Z=695.410
pol=5 vert=45 dist=5737.673757 XY=(622717.356, 2112377.422) 

 ### <span style="color:#cc416d">11. Recuperar la distancia desde el XY reconstruido</span>

In [13]:
errores_distancia = []
errores_z = []

for registro in vertices_reconstruidos:

    punto_xy_geom = QgsGeometry.fromPointXY(
        QgsPointXY(
            registro["x_utm"],
            registro["y_utm"]
        )
    )

    distancia_recuperada = (
        geom_guia_simple.lineLocatePoint(
            punto_xy_geom
        )
    )

    error_distancia = (
        distancia_recuperada
        - registro["distancia_2d"]
    )

    error_z = (
        registro["z_3d"]
        - registro["elevacion_2d"]
    )

    registro["distancia_recuperada"] = (
        distancia_recuperada
    )

    registro["error_distancia"] = (
        error_distancia
    )

    registro["error_z"] = error_z

    errores_distancia.append(
        error_distancia
    )

    errores_z.append(
        error_z
    )

 ### <span style="color:#cc416d">Resumen</span>

In [14]:
import numpy as np

errores_distancia_np = np.array(
    errores_distancia,
    dtype=float
)

errores_z_np = np.array(
    errores_z,
    dtype=float
)

print("Vértices evaluados:", len(errores_distancia_np))

print("\nError de distancia:")
print(
    "Mínimo:",
    errores_distancia_np.min()
)
print(
    "Máximo:",
    errores_distancia_np.max()
)
print(
    "Promedio absoluto:",
    np.mean(
        np.abs(errores_distancia_np)
    )
)
print(
    "RMSE:",
    np.sqrt(
        np.mean(
            errores_distancia_np ** 2
        )
    )
)

print("\nError Z:")
print("Mínimo:", errores_z_np.min())
print("Máximo:", errores_z_np.max())
print(
    "Promedio absoluto:",
    np.mean(np.abs(errores_z_np))
)

Vértices evaluados: 1182

Error de distancia:
Mínimo: -1.0504663805477321e-10
Máximo: 1.0732037480920553e-10
Promedio absoluto: 2.393961622282506e-11
RMSE: 3.569706452512475e-11

Error Z:
Mínimo: 0.0
Máximo: 0.0
Promedio absoluto: 0.0


<h2>Resultados de la reconstrucción espacial</h2>

<p>
Se reconstruyeron en coordenadas espaciales los vértices de los cinco polígonos
geológicos interpretados en el perfil 2D. En total se evaluaron
<strong>1,182 vértices</strong>.
</p>

<p>
Para cada vértice se utilizó la coordenada X del perfil como distancia acumulada
sobre la línea guía. Posteriormente, dicha distancia se transformó nuevamente a
coordenadas XY mediante:
</p>

<p style="text-align:center;">
    <code>geom_guia_simple.interpolate(distancia)</code>
</p>

<p>
La coordenada Y del perfil 2D se conservó como elevación Z en la geometría
reconstruida.
</p>

<h3>Prueba de ida y vuelta</h3>

<p>
Después de reconstruir cada posición XY, el punto fue proyectado nuevamente sobre
la línea guía mediante <code>lineLocatePoint()</code>. La distancia recuperada se
comparó con la distancia original almacenada en el perfil 2D.
</p>

<table style="border-collapse:collapse; margin-top:10px;">
    <thead>
        <tr>
            <th style="border:1px solid #999; padding:6px;">Indicador</th>
            <th style="border:1px solid #999; padding:6px;">Valor</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border:1px solid #999; padding:6px;">
                Error mínimo de distancia
            </td>
            <td style="border:1px solid #999; padding:6px;">
                −1.05 × 10<sup>−10</sup> m
            </td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:6px;">
                Error máximo de distancia
            </td>
            <td style="border:1px solid #999; padding:6px;">
                1.07 × 10<sup>−10</sup> m
            </td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:6px;">
                Error medio absoluto
            </td>
            <td style="border:1px solid #999; padding:6px;">
                2.39 × 10<sup>−11</sup> m
            </td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:6px;">
                RMSE de distancia
            </td>
            <td style="border:1px solid #999; padding:6px;">
                3.57 × 10<sup>−11</sup> m
            </td>
        </tr>
    </tbody>
</table>

<h3>Conservación de la elevación</h3>

<p>
La elevación del perfil 2D se utilizó directamente como coordenada Z en la
reconstrucción tridimensional. Las diferencias obtenidas fueron:
</p>

<table style="border-collapse:collapse; margin-top:10px;">
    <thead>
        <tr>
            <th style="border:1px solid #999; padding:6px;">Indicador</th>
            <th style="border:1px solid #999; padding:6px;">Valor</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border:1px solid #999; padding:6px;">
                Error Z mínimo
            </td>
            <td style="border:1px solid #999; padding:6px;">0.0 m</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:6px;">
                Error Z máximo
            </td>
            <td style="border:1px solid #999; padding:6px;">0.0 m</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:6px;">
                Error Z medio absoluto
            </td>
            <td style="border:1px solid #999; padding:6px;">0.0 m</td>
        </tr>
    </tbody>
</table>

<div style="
border-left:6px solid #2E75B6;
background:#F4F8FC;
padding:14px;
margin-top:18px;
">

<h3 style="margin-top:0;">Conclusión</h3>

<div>
La transformación de las distancias del perfil 2D a coordenadas XY mediante
<code>QgsGeometry.interpolate()</code> conserva la posición espacial con errores
únicamente atribuibles a la precisión numérica del sistema.
</div>

<div style="margin-top:12px;">
La coordenada Z se conserva sin ninguna modificación. En consecuencia, el posible
desfase horizontal observado en el producto final no se origina durante la
reconstrucción espacial del perfil 3D.
</div>

<div style="margin-top:12px;">
Las siguientes etapas que deben revisarse son la generación de contactos geológicos,
el recorte de segmentos, la construcción de polígonos finales y la comparación con
la salida producida directamente por SecGeol.
</div>

</div>